In [ ]:
# =====================================================================================
# Task 1  Baseline: Smoothed N-gram Language Model (WikiText-2)
#   Step 1  tokenize, lowercase, 30k vocabulary, <UNK>
#   Step 2  count tables for 1-, 2-, 3- and 4-grams
#   Step 3  Add-k smoothing {k=1, 0.5, 0.1}
#   Step 4  Good–Turing smoothing (counts r <= 5 re-estimated)
#   Step 5  interpolated Kneser–Ney, d = 0.75, λ tuned on validation over {0.1, ..., 0.9}
#   Step 6  perplexity of all five variants on the test split
#   Step 7  BLiMP accuracy (per paradigm + overall) for the best variant by validation perplexity
# All five variants are 4-gram models, so the smoothing method is the only difference between them.
# Seeds: Task 1 has no randomness (pure counting, fixed BLiMP paradigm list), so no seed is needed.
# =====================================================================================
import json
import math
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

# ---- Configuration -------------------------------------------------------------------
DATA_DIR = Path("data/wikitext2/wikitext-2-raw-v1")
RESULTS_DIR = Path("results/task1")
ORDER = 4                       # highest n-gram order
VOCAB_SIZE = 30_000             # keep the 30,000 most frequent tokens ...
MIN_COUNT = 2                   # ... among those seen at least twice in training
ADD_K_VALUES = [1, 0.5, 0.1]
GT_MAX_COUNT = 5                # Good–Turing re-estimates counts r <= 5
KN_DISCOUNT = 0.75
KN_LAMBDAS = [round(0.1 * i, 1) for i in range(1, 10)]     # 0.1, 0.2, ..., 0.9
BOS, EOS, UNK = "<s>", "</s>", "<UNK>"

# BLiMP (official release, pinned): 10 of the 67 paradigms, spanning morphology, syntax and
# semantics, one grammatical phenomenon each. The same list is reused unchanged in Tasks 2 and 3.
BLIMP_REPO = "nyu-mll/blimp"
BLIMP_REV = "877fba0801ffb7cbd8c39c1ff314a46f053f6036"
BLIMP_PARADIGMS = {
    "determiner_noun_agreement_1": "morphology",
    "regular_plural_subject_verb_agreement_1": "morphology",
    "irregular_past_participle_verbs": "morphology",
    "anaphor_number_agreement": "morphology",
    "wh_vs_that_no_gap": "syntax",
    "adjunct_island": "syntax",
    "passive_1": "syntax",
    "existential_there_quantifiers_1": "semantics",
    "superlative_quantifiers_1": "semantics",
    "sentential_negation_npi_licensor_present": "semantics",
}
BLIMP_TIE_TOL = 1e-9            # score differences below this are floating-point noise, i.e. a tie

WIKITEXT_REPO = "Salesforce/wikitext"       # official Hugging Face mirror of WikiText-2, pinned to a fixed commit
WIKITEXT_REV = "b08601e04326c79dfdd32d625aee71d232d685c3"


def download_wikitext2_raw():
    """Fetch WikiText-2 (raw) into DATA_DIR. Same files as download_data.py, inlined so this cell also runs on
    Google Colab, where download_data.py does not exist."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for hf_split, name in (("train", "train"), ("validation", "valid"), ("test", "test")):
        path = hf_hub_download(WIKITEXT_REPO, f"wikitext-2-raw-v1/{hf_split}-00000-of-00001.parquet",
                               repo_type="dataset", revision=WIKITEXT_REV)
        lines = pq.read_table(path).column("text").to_pylist()
        (DATA_DIR / f"wiki.{name}.raw").write_text("".join(line or "\n" for line in lines), encoding="utf-8")


RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not (DATA_DIR / "wiki.train.raw").exists():
    download_wikitext2_raw()
t_start = time.time()

# ---- Step 1: tokenize, lowercase, build the vocabulary, map rare/unknown tokens to <UNK> --
TOKEN_RE = re.compile(r"'\w+|\w+|[^\w\s]")


def tokenize(text):
    """Punctuation-aware tokenizer: lowercase, then split into words, clitics ('s, 't, ...) and
    single punctuation marks. WikiText writes in-word punctuation as @-@, @,@ and @.@; these are
    turned back into - , . first, so WikiText and ordinary text (BLiMP) are tokenized the same way."""
    text = text.lower().replace("@-@", "-").replace("@,@", ",").replace("@.@", ".")
    return TOKEN_RE.findall(text)


def read_split(name):
    """One token list per non-empty line (each WikiText line is a paragraph or a heading)."""
    with open(DATA_DIR / f"wiki.{name}.raw", encoding="utf-8") as f:
        return [tokenize(line) for line in f if line.strip()]


raw = {split: read_split(split) for split in ("train", "valid", "test")}
freq = Counter(tok for line in raw["train"] for tok in line)
frequent = sorted((w for w, c in freq.items() if c >= MIN_COUNT), key=lambda w: (-freq[w], w))
vocab = set(frequent[:VOCAB_SIZE])          # ties in frequency broken alphabetically
V = len(vocab) + 2                          # tokens a model can predict: vocabulary + <UNK> + </s>


def to_vocab(tokens):
    return [tok if tok in vocab else UNK for tok in tokens]


data = {split: [to_vocab(line) for line in lines] for split, lines in raw.items()}

print("Step 1 — tokenization and vocabulary")
print(f"  train types: {len(freq):,}   types seen >= {MIN_COUNT} times: {len(frequent):,}   "
      f"vocabulary kept: {len(vocab):,}   predictable tokens V = {V:,} (incl. {UNK}, {EOS})")
for split, lines in data.items():
    n_tok = sum(len(line) for line in lines)
    n_unk = sum(tok == UNK for line in lines for tok in line)
    print(f"  {split:5s}: {len(lines):6,} lines  {n_tok:9,} tokens  {UNK} rate {n_unk / n_tok:.2%}")


# ---- Step 2: count tables for 1-, 2-, 3- and 4-grams -----------------------------------
def contexts_and_targets(tokens):
    """Yield (previous ORDER-1 tokens, token) for every token of a line and the final </s>.
    The line is padded with <s> on the left so every token has a full history."""
    seq = [BOS] * (ORDER - 1) + tokens + [EOS]
    for i in range(ORDER - 1, len(seq)):
        yield tuple(seq[i - ORDER + 1:i]), seq[i]


counts = {n: Counter() for n in range(1, ORDER + 1)}      # counts[n][(w1, ..., wn)]
for line in data["train"]:
    for context, word in contexts_and_targets(line):
        for n in range(1, ORDER + 1):
            counts[n][context[ORDER - n:] + (word,)] += 1


def context_stats(table):
    """For every context h: c(h) = sum_w c(h, w), and N1+(h •) = number of distinct w seen after h."""
    totals, types = Counter(), Counter()
    for gram, c in table.items():
        totals[gram[:-1]] += c
        types[gram[:-1]] += 1
    return totals, types


top_totals, _ = context_stats(counts[ORDER])
print("Step 2 — count tables")
for n in range(1, ORDER + 1):
    print(f"  {n}-grams: {len(counts[n]):10,} distinct, {sum(counts[n].values()):,} total")


# ---- Step 3: Add-k smoothing -------------------------------------------------------------
class AddK:
    """P(w | h) = (c(h, w) + k) / (c(h) + k * V): every possible 4-gram is treated as seen k extra times."""

    def __init__(self, table, totals, k):
        self.table, self.totals, self.k = table, totals, k

    def prob(self, context, word):
        return (self.table.get(context + (word,), 0) + self.k) / (self.totals.get(context, 0) + self.k * V)


# ---- Step 4: Good–Turing smoothing -------------------------------------------------------
class GoodTuring:
    """Counts r <= GT_MAX_COUNT are replaced by r* = (r + 1) * N_{r+1} / N_r (N_r = number of distinct
    4-grams seen exactly r times); larger counts are kept. P(w | h) = r*(h, w) / c(h) for seen 4-grams.
    The probability this frees in a context is shared equally by the words never seen after it.
    In the few contexts where every continuation has r > GT_MAX_COUNT nothing is freed; there, unseen
    words get Good–Turing's r = 0 estimate r*_0 = N_1 / N_0 and the context is renormalized.
    Contexts never seen in training get the uniform distribution 1 / V."""

    def __init__(self, table, totals, max_count):
        n_r = Counter(table.values())
        self.r_star = {r: (r + 1) * n_r[r + 1] / n_r[r] for r in range(1, max_count + 1)}
        r_star_0 = n_r[1] / (V ** ORDER - len(table))       # N_0 = number of possible 4-grams never seen
        self.table, self.totals = table, totals
        seen_mass, n_seen = defaultdict(float), Counter()
        for gram, c in table.items():
            seen_mass[gram[:-1]] += self.r_star.get(c, c)
            n_seen[gram[:-1]] += 1
        self.seen_scale, self.unseen_prob, self.no_leftover = {}, {}, []
        for h, total in totals.items():
            leftover = total - seen_mass[h]
            if leftover > 1e-9:
                self.seen_scale[h] = 1 / total
                self.unseen_prob[h] = leftover / total / (V - n_seen[h])
            else:
                z = seen_mass[h] + (V - n_seen[h]) * r_star_0
                self.seen_scale[h] = 1 / z
                self.unseen_prob[h] = r_star_0 / z
                self.no_leftover.append(h)

    def prob(self, context, word):
        if context not in self.totals:
            return 1 / V
        c = self.table.get(context + (word,), 0)
        if c:
            return self.r_star.get(c, c) * self.seen_scale[context]
        return self.unseen_prob[context]


# ---- Step 5: interpolated Kneser–Ney -----------------------------------------------------
class KneserNey:
    """Interpolated Kneser–Ney over orders 1..ORDER with absolute discount d.
    The 4-gram level uses raw counts; lower levels use continuation counts N1+(• h w), the number of
    distinct words seen right before "h w". The unigram level is P_1(w) = N1+(• w) / N1+(• •).
    For n = 2..ORDER, with h' = h minus its first word:
        P_KN,n(w | h) = max(c(h, w) - d, 0) / c(h) + d * N1+(h •) / c(h) * P_{n-1}(w | h')
        P_n(w | h)    = λ * P_KN,n(w | h) + (1 - λ) * P_{n-1}(w | h')
    The first line is standard interpolated KN (higher-order estimate), the second interpolates it
    with the lower-order estimate using the weight λ tuned on validation. If c(h) = 0, P_n = P_{n-1}.
    Both lines mix probability distributions, so every P_n sums to 1."""

    def __init__(self, counts, d, lam):
        self.d, self.lam = d, lam
        self.tables = {ORDER: counts[ORDER]}
        for n in range(1, ORDER):
            continuation = Counter()
            for gram in counts[n + 1]:
                continuation[gram[1:]] += 1
            self.tables[n] = continuation
        self.stats = {n: context_stats(table) for n, table in self.tables.items()}
        self.unigram_total = self.stats[1][0][()]

    def prob(self, context, word):
        p = self.tables[1].get((word,), 0) / self.unigram_total
        for n in range(2, ORDER + 1):
            h = context[ORDER - n:]
            totals, types = self.stats[n]
            total = totals.get(h, 0)
            if total:
                c = self.tables[n].get(h + (word,), 0)
                p_kn = (max(c - self.d, 0) + self.d * types[h] * p) / total
                p = self.lam * p_kn + (1 - self.lam) * p
        return p


# ---- Step 6: perplexity ------------------------------------------------------------------
def log_probs(model, tokens):
    return [math.log(model.prob(context, word)) for context, word in contexts_and_targets(tokens)]


def perplexity(model, lines):
    """PPL = exp(-(1/N) * sum_i log P(w_i | context)); N counts every token plus one </s> per line."""
    total, n = 0.0, 0
    for line in lines:
        lps = log_probs(model, line)
        total += sum(lps)
        n += len(lps)
    return math.exp(-total / n)


kn = KneserNey(counts, KN_DISCOUNT, lam=None)
lambda_grid = {}
for lam in KN_LAMBDAS:
    kn.lam = lam
    lambda_grid[lam] = perplexity(kn, data["valid"])
best_lambda = min(lambda_grid, key=lambda_grid.get)
kn.lam = best_lambda
print(f"Step 5 — Kneser–Ney λ grid search on validation: best λ = {best_lambda} "
      f"(validation PPL {lambda_grid[best_lambda]:.2f})")

models = {f"Add-k (k={k})": AddK(counts[ORDER], top_totals, k) for k in ADD_K_VALUES}
models["Good–Turing"] = GoodTuring(counts[ORDER], top_totals, GT_MAX_COUNT)
models[f"Kneser–Ney (d={KN_DISCOUNT}, λ={best_lambda})"] = kn

# Sanity check: every variant must give a proper distribution over the V predictable tokens,
# for a frequent context, the sentence-start context, a Good–Turing "no leftover" context and an
# unseen context.
all_words = sorted(vocab) + [UNK, EOS]
check_contexts = [("one", "of", "the"), (BOS, BOS, BOS), models["Good–Turing"].no_leftover[0], (EOS, EOS, EOS)]
for name, model in models.items():
    for h in check_contexts:
        total_prob = math.fsum(model.prob(h, w) for w in all_words)
        assert abs(total_prob - 1) < 1e-6, (name, h, total_prob)
print(f"  sanity check passed: all 5 variants sum to 1 over V for {len(check_contexts)} test contexts")

ppl = {name: {"valid": perplexity(m, data["valid"]), "test": perplexity(m, data["test"])}
       for name, m in models.items()}
best_variant = min(ppl, key=lambda name: ppl[name]["valid"])
print(f"Step 6 — perplexity computed; best variant by validation PPL: {best_variant}")


# ---- Step 7: BLiMP accuracy for the best variant -----------------------------------------
def load_blimp():
    """{paradigm: list of {"sentence_good", "sentence_bad"}} for the selected paradigms."""
    pairs = {}
    for name in BLIMP_PARADIGMS:
        path = hf_hub_download(BLIMP_REPO, f"{name}/train-00000-of-00001.parquet",
                               repo_type="dataset", revision=BLIMP_REV)
        pairs[name] = pq.read_table(path, columns=["sentence_good", "sentence_bad"]).to_pylist()
    return pairs


def sentence_score(model, sentence):
    """Length-normalized log-probability: mean log P over the sentence's tokens and its </s>,
    after the same tokenization, lowercasing and <UNK> mapping as the training data."""
    lps = log_probs(model, to_vocab(tokenize(sentence)))
    return sum(lps) / len(lps)


def blimp_accuracy(model, blimp):
    """A pair is correct if the model prefers the grammatical sentence (higher score by more than
    BLIMP_TIE_TOL). Ties — e.g. both sentences identical after <UNK> mapping, or a reordering the
    model cannot see because it backs off to unigrams — express no preference and count as not correct."""
    rows = []
    for name, pairs in blimp.items():
        diffs = [sentence_score(model, p["sentence_good"]) - sentence_score(model, p["sentence_bad"])
                 for p in pairs]
        correct = sum(d > BLIMP_TIE_TOL for d in diffs)
        rows.append({"paradigm": name, "category": BLIMP_PARADIGMS[name], "pairs": len(pairs),
                     "correct": correct, "ties": sum(abs(d) <= BLIMP_TIE_TOL for d in diffs),
                     "accuracy": correct / len(pairs)})
    table = pd.DataFrame(rows)
    overall = table["correct"].sum() / table["pairs"].sum()
    return table, overall


blimp = load_blimp()
blimp_table, blimp_overall = blimp_accuracy(models[best_variant], blimp)

# ---- Results -------------------------------------------------------------------------------
ppl_table = pd.DataFrame([{"variant": name, "validation PPL": v["valid"], "test PPL": v["test"]}
                          for name, v in ppl.items()])
lambda_table = pd.DataFrame({"λ": list(lambda_grid), "validation PPL": list(lambda_grid.values())})

print(f"\nTable 1.1 — Perplexity of the five smoothing variants (4-gram, WikiText-2)")
print(ppl_table.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
print(f"\nTable 1.2 — Kneser–Ney λ grid search (validation split)")
print(lambda_table.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
print(f"\nTable 1.3 — BLiMP accuracy, best variant: {best_variant}")
print(blimp_table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"Overall BLiMP accuracy: {blimp_overall:.3f} "
      f"({blimp_table['correct'].sum():,} / {blimp_table['pairs'].sum():,} pairs)")

results = {
    "vocab_size": len(vocab),
    "predictable_tokens_V": V,
    "order": ORDER,
    "kn_discount": KN_DISCOUNT,
    "kn_lambda_grid_valid_ppl": lambda_grid,
    "kn_best_lambda": best_lambda,
    "perplexity": ppl,
    "best_variant": best_variant,
    "blimp_per_paradigm": dict(zip(blimp_table["paradigm"], blimp_table["accuracy"])),
    "blimp_ties_per_paradigm": dict(zip(blimp_table["paradigm"], blimp_table["ties"].tolist())),
    "blimp_overall": blimp_overall,
}
(RESULTS_DIR / "results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False))
ppl_table.to_csv(RESULTS_DIR / "perplexity.csv", index=False)
lambda_table.to_csv(RESULTS_DIR / "kn_lambda_grid.csv", index=False)
blimp_table.to_csv(RESULTS_DIR / "blimp.csv", index=False)
print(f"\nResults written to {RESULTS_DIR}/  (total time {time.time() - t_start:.0f}s)")


Step 1 — tokenization and vocabulary
  train types: 66,081   types seen >= 2 times: 39,189   vocabulary kept: 30,000   predictable tokens V = 30,002 (incl. <UNK>, </s>)
  train: 23,767 lines  2,060,762 tokens  <UNK> rate 2.20%
  valid:  2,461 lines    214,970 tokens  <UNK> rate 4.67%
  test :  2,891 lines    242,089 tokens  <UNK> rate 5.58%
Step 2 — count tables
  1-grams:     30,002 distinct, 2,084,529 total
  2-grams:    596,064 distinct, 2,084,529 total
  3-grams:  1,388,276 distinct, 2,084,529 total
  4-grams:  1,808,094 distinct, 2,084,529 total
Step 5 — Kneser–Ney λ grid search on validation: best λ = 0.8 (validation PPL 243.95)
  sanity check passed: all 5 variants sum to 1 over V for 4 test contexts
Step 6 — perplexity computed; best variant by validation PPL: Kneser–Ney (d=0.75, λ=0.8)

Table 1.1 — Perplexity of the five smoothing variants (4-gram, WikiText-2)
                   variant  validation PPL  test PPL
               Add-k (k=1)       21,546.98 21,259.14
            

### Task 1 — Discussion

**Perplexity (Table 1.1).** Kneser–Ney (test PPL **233.3**) beats the other four variants by a factor of 40–90. The cause is data sparsity at n = 4: 93.6% of the distinct training 4-grams occur only once, and 83.9% of test tokens come with a 4-gram never seen in training (for 58.2% even the 3-word context is unseen). Add-k and Good–Turing only *reserve* probability for unseen 4-grams and then spread it uniformly over all 30,002 tokens, so for most test tokens they cannot use the shorter context that *was* observed (e.g. "of the" instead of "most of the"). Add-k is worst because it adds k·V ≈ 30,000·k pseudo-counts to contexts whose median count is 1, which drowns the real counts; a smaller k helps (21,259 → 15,894) but cannot fix this. Good–Turing discounts more sensibly (singletons get r\*(1) = 0.09) and lowers perplexity to 9,567, but it has the same uniform-fallback problem. Kneser–Ney interpolates with the 3-, 2- and 1-gram estimates (using continuation counts), which is exactly the information unseen 4-grams need. The λ search (Table 1.2) shows that too much weight on lower orders hurts (λ = 0.1 → 452), with the optimum at λ = 0.8.

**BLiMP (Table 1.3).** Overall accuracy is **62.9%**, above chance (50%) and close to the 61.2% the BLiMP paper reports for a 5-gram model. The model succeeds when the grammatical difference is visible inside a few adjacent words: sentential_negation_npi_licensor_present 99.6% ("not ever" vs "probably ever"), wh_vs_that_no_gap 96.3%, existential_there_quantifiers_1 87.3% ("there was a" vs "there was each"). It fails in three ways:
1. **No evidence to separate the two words (ties).** In 86 pairs of determiner_noun_agreement_1 and 31 of irregular_past_participle_verbs the two sentences become identical after `<UNK>` mapping; other ties come from rare words with identical counts (e.g. "shrank" / "shrunk", 4 occurrences each). A tie is not a preference, so it counts as incorrect.
2. **Word order outside observed n-grams.** adjunct_island (29.5%, 387 ties) reorders the same words; where the model backs off to unigrams, the score no longer depends on order.
3. **Dependencies beyond the 4-gram window.** anaphor_number_agreement (37.0%) requires matching the reflexive (herself / themselves) to the subject, which the model cannot connect to it, so it decides on local word statistics and ends up below chance.

**Implementation choices (to state in the report).**
- *Tokenizer:* regex `'\w+|\w+|[^\w\s]` after lowercasing; WikiText's `@-@`, `@,@`, `@.@` are mapped back to `-`, `,`, `.` so WikiText and BLiMP are tokenized identically. Each WikiText line is one sequence, padded with `<s>` and ended with `</s>`; `</s>` is counted in N.
- *Model order:* all five variants are 4-gram models, so only the smoothing differs.
- *Good–Turing:* the mass freed by discounting r ≤ 5 is shared equally by the words unseen after a context; in the 1,718 of 1.38M contexts where every continuation has r > 5 (nothing freed), unseen words get Good–Turing's r = 0 estimate N₁/N₀ — otherwise 89 test tokens would get probability 0 and perplexity would be infinite.
- *Kneser–Ney λ:* textbook interpolated KN computes its back-off weight from d; to tune λ as required, each order mixes the KN estimate with the lower-order estimate, P_n = λ·P_KN,n + (1 − λ)·P_{n−1}. λ = 1 is exactly textbook KN; every P_n sums to 1.
- *BLiMP:* 10 paradigms chosen by us (4 morphology, 3 syntax, 3 semantics, one phenomenon each) from the official `nyu-mll/blimp` release; score differences below 1e-9 are ties and count as incorrect.
- *Seeds:* Task 1 has no randomness, so no seed is needed.

In [ ]:
# =====================================================================================
# Task 2 — Paper Reproduction: AWD-LSTM (Merity, Keskar & Socher 2017, arXiv:1708.02182)
#   Step 1  3-layer LSTM, embedding size 400, hidden size 1150, input embedding tied to the softmax
#   Step 2  DropConnect p=0.5 on hidden-to-hidden weights, embedding dropout p=0.1, output dropout p=0.4
#   Step 3  NT-ASGD optimizer, initial learning rate 30
#   Step 4  batch size 80, BPTT length 70, gradient clipping at norm 0.25
#   Step 5  train for 50 epochs (the paper trains up to 500 — deviation, see the discussion)
#   Step 6  test perplexity + BLiMP (same 10 paradigms and same scoring as Task 1)
# Uses Task 1's tokenizer, vocabulary and splits (run the Task 1 cell first): both models predict exactly
# the same 244,980 test tokens, so perplexities are directly comparable and Task 3 can mix them token by token.
# Seeds: SEED = 1111 for random / numpy / torch (model initialization); epoch e uses torch seed SEED + e for
# its dropout masks. GPU kernels (CUDA / Apple MPS) are not guaranteed to be bit-for-bit deterministic.
# Training takes about 7–8 hours on an Apple M3 (MPS, ~9 min per epoch). A checkpoint is written after every
# epoch: re-running this cell resumes from it, and once all epochs are done the cell only evaluates.
# =====================================================================================
import random
import warnings
from contextlib import contextmanager

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call

# ---- Configuration -------------------------------------------------------------------
TASK2_DIR = Path("results/task2")
CKPT_DIR = Path("checkpoints")
SEED = 1111
EMB_SIZE, HIDDEN_SIZE, N_LAYERS = 400, 1150, 3
DROPCONNECT, EMB_DROPOUT, OUT_DROPOUT = 0.5, 0.1, 0.4
LR = 30
NONMONO = 5                     # NT-ASGD non-monotone interval n (paper: n = 5)
BATCH_SIZE, BPTT, CLIP = 80, 70, 0.25
EPOCHS = 50
EVAL_BATCH_SIZE = 10            # validation after each epoch (as in the official code); final evaluation uses batch size 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
LAST_CKPT = CKPT_DIR / f"task2_awd_lstm_h{HIDDEN_SIZE}_last.pt"
BEST_CKPT = CKPT_DIR / f"task2_awd_lstm_h{HIDDEN_SIZE}_best.pt"


def free_device_cache():
    """Return cached GPU memory, e.g. from an earlier run of this cell, so a re-run does not run out of memory."""
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()


free_device_cache()
TASK2_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
# DropConnect swaps in a new hidden-to-hidden matrix each call; on CUDA this makes PyTorch warn every step
warnings.filterwarnings("ignore", message="RNN module weights are not part of single contiguous chunk")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
t_start = time.time()

# ---- Data: Task 1's tokens as one id stream per split ------------------------------------
itos = sorted(vocab) + [UNK, EOS]           # the V = 30,002 predictable tokens, same order as Task 1's all_words
stoi = {w: i for i, w in enumerate(itos)}
EOS_ID = stoi[EOS]


def to_ids(lines):
    """</s>, then every line followed by </s>. Predicting positions 1, 2, ... gives exactly Task 1's targets
    (every token plus one </s> per line); the leading </s> is the input for the very first token."""
    ids = [EOS_ID]
    for line in lines:
        ids.extend(stoi[tok] for tok in line)
        ids.append(EOS_ID)
    return torch.tensor(ids, dtype=torch.long)


def batchify(ids, bsz):
    """Cut the stream into bsz contiguous columns -> tensor (length, bsz); the remainder is dropped."""
    n = ids.size(0) // bsz
    return ids[:n * bsz].view(bsz, n).t().contiguous().to(DEVICE)


def get_batch(source, i):
    seq_len = min(BPTT, source.size(0) - 1 - i)
    return source[i:i + seq_len], source[i + 1:i + 1 + seq_len]


ids = {split: to_ids(lines) for split, lines in data.items()}
train_data = batchify(ids["train"], BATCH_SIZE)
valid_data = batchify(ids["valid"], EVAL_BATCH_SIZE)


# ---- Steps 1 + 2: AWD-LSTM with its three regularizers -----------------------------------
def embedding_dropout(embedding, words, p, training):
    """Drop whole words: each row of the embedding matrix (one word type) is zeroed with probability p
    for this batch — every occurrence of that word — and the kept rows are scaled by 1 / (1 - p)."""
    weight = embedding.weight
    if training and p > 0:
        mask = weight.new_empty(weight.size(0), 1).bernoulli_(1 - p) / (1 - p)
        weight = weight * mask
    return F.embedding(words, weight)


def locked_dropout(x, p, training):
    """Variational dropout on x (time, batch, features): one mask per sequence, shared by all time steps."""
    if not training or p == 0:
        return x
    mask = x.new_empty(1, x.size(1), x.size(2)).bernoulli_(1 - p) / (1 - p)
    return x * mask


class AWDLSTM(nn.Module):
    def __init__(self, n_tokens):
        super().__init__()
        self.embedding = nn.Embedding(n_tokens, EMB_SIZE)
        # 400 -> 1150 -> 1150 -> 400: the last layer outputs EMB_SIZE so the softmax can share the embedding matrix
        sizes = [EMB_SIZE] + [HIDDEN_SIZE] * (N_LAYERS - 1) + [EMB_SIZE]
        self.lstms = nn.ModuleList(nn.LSTM(sizes[i], sizes[i + 1]) for i in range(N_LAYERS))
        self.decoder = nn.Linear(EMB_SIZE, n_tokens)
        self.decoder.weight = self.embedding.weight          # weight tying
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)   # as in the official code
        nn.init.zeros_(self.decoder.bias)

    def init_hidden(self, bsz):
        return [(torch.zeros(1, bsz, lstm.hidden_size, device=DEVICE),
                 torch.zeros(1, bsz, lstm.hidden_size, device=DEVICE)) for lstm in self.lstms]

    def forward(self, words, hidden):
        x = embedding_dropout(self.embedding, words, EMB_DROPOUT, self.training)
        new_hidden = []
        for lstm, h in zip(self.lstms, hidden):
            # DropConnect: a new mask on the hidden-to-hidden weights for every forward pass, so the same
            # connections are dropped at all time steps of the batch
            w_hh = F.dropout(lstm.weight_hh_l0, DROPCONNECT, self.training)
            x, h = functional_call(lstm, {"weight_hh_l0": w_hh}, (x, h))
            new_hidden.append(h)
        x = locked_dropout(x, OUT_DROPOUT, self.training)
        return self.decoder(x), new_hidden


model = AWDLSTM(V).to(DEVICE)
params = list(model.parameters())           # the tied embedding / softmax matrix appears once
print(f"Steps 1–2 — AWD-LSTM on {DEVICE}: layers {EMB_SIZE}->{HIDDEN_SIZE}->{HIDDEN_SIZE}->{EMB_SIZE}, "
      f"tied weights, {sum(p.numel() for p in params) / 1e6:.1f}M parameters")


# ---- Steps 3–5: NT-ASGD training ---------------------------------------------------
def detach(hidden):
    return [(h.detach(), c.detach()) for h, c in hidden]


def evaluate_loss(source, bsz):
    """Mean cross-entropy per token on a batchified split, dropout off."""
    model.eval()
    hidden, total, n = model.init_hidden(bsz), 0.0, 0
    with torch.no_grad():
        for i in range(0, source.size(0) - 1, BPTT):
            x, y = get_batch(source, i)
            logits, hidden = model(x, hidden)
            total += F.cross_entropy(logits.view(-1, V), y.reshape(-1), reduction="sum").item()
            n += y.numel()
    return total / n


@contextmanager
def averaged_weights(avg):
    """Temporarily load the NT-ASGD averaged weights into the model (no-op before averaging has started)."""
    if avg is None:
        yield
        return
    backup = [p.detach().clone() for p in params]
    with torch.no_grad():
        for p, a in zip(params, avg):
            p.copy_(a)
    try:
        yield
    finally:
        with torch.no_grad():
            for p, b in zip(params, backup):
                p.copy_(b)


def train_epoch(optimizer, avg, n_avg):
    model.train()
    hidden, total, n = model.init_hidden(BATCH_SIZE), 0.0, 0
    for i in range(0, train_data.size(0) - 1, BPTT):
        x, y = get_batch(train_data, i)
        hidden = detach(hidden)             # truncated BPTT: carry the state over, cut the gradient
        logits, hidden = model(x, hidden)
        loss = F.cross_entropy(logits.view(-1, V), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(params, CLIP)
        optimizer.step()
        if avg is not None:                 # averaging phase: running mean of every iterate since the switch
            n_avg += 1
            with torch.no_grad():
                for a, p in zip(avg, params):
                    a.add_(p - a, alpha=1 / n_avg)
        # .item() waits for the GPU every step; without it the CPU queues steps faster than an Apple-MPS GPU
        # runs them and memory grows by one batch of logits per queued step until it runs out
        total += loss.item() * y.numel()
        n += y.numel()
    return total / n, n_avg


def save_checkpoint(obj, path):
    """Write to a temporary file, then rename it into place, so stopping the cell during a save can never
    leave a half-written checkpoint behind."""
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


# Plain SGD until the trigger fires, then SGD + iterate averaging — with no momentum and a constant
# learning rate this is exactly ASGD as used in the paper (torch.optim.ASGD with t0 = 0, lambd = 0).
optimizer = torch.optim.SGD(params, lr=LR)
state = {"history": [], "avg": None, "n_avg": 0, "switch_epoch": None, "best_valid_loss": float("inf")}
if LAST_CKPT.exists():
    ckpt = torch.load(LAST_CKPT, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    state = ckpt["state"]
    del ckpt                                # its copy of the weights is no longer needed
    print(f"  resuming from {LAST_CKPT} (epochs done: {len(state['history'])})")

for epoch in range(len(state["history"]) + 1, EPOCHS + 1):
    torch.manual_seed(SEED + epoch)
    t0 = time.time()
    averaging = state["avg"] is not None
    train_loss, state["n_avg"] = train_epoch(optimizer, state["avg"], state["n_avg"])
    with averaged_weights(state["avg"]):
        valid_loss = evaluate_loss(valid_data, EVAL_BATCH_SIZE)
        if valid_loss < state["best_valid_loss"]:
            state["best_valid_loss"] = valid_loss
            save_checkpoint(model.state_dict(), BEST_CKPT)
    # NT-ASGD trigger (paper, Algorithm 1): start averaging once the validation loss is worse than the
    # best validation loss from more than NONMONO epochs ago
    past = [h["valid_loss"] for h in state["history"]]
    switched = state["avg"] is None and len(past) > NONMONO and valid_loss > min(past[:-NONMONO])
    if switched:
        state["avg"], state["n_avg"], state["switch_epoch"] = [p.detach().clone() for p in params], 1, epoch
    state["history"].append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss,
                             "train_ppl": math.exp(train_loss), "valid_ppl": math.exp(valid_loss),
                             "averaged": averaging, "minutes": (time.time() - t0) / 60})
    save_checkpoint({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "state": state}, LAST_CKPT)
    print(f"  epoch {epoch:3d} | train PPL {math.exp(train_loss):8.2f} | valid PPL {math.exp(valid_loss):8.2f} | "
          f"{'ASGD' if averaging else 'SGD '} | {(time.time() - t0) / 60:4.1f} min"
          + ("  <- NT-ASGD trigger: averaging from the next epoch on" if switched else ""), flush=True)

history = pd.DataFrame(state["history"])
best_epoch = int(history.loc[history["valid_loss"].idxmin(), "epoch"])
print(f"Steps 3–5 — trained {len(history)} epochs ({history['minutes'].sum() / 60:.1f} h); NT-ASGD switch: "
      f"{'after epoch ' + str(state['switch_epoch']) if state['switch_epoch'] else 'never triggered'}; "
      f"best validation loss at epoch {best_epoch}")
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE, weights_only=True))
free_device_cache()


# ---- Step 6: test perplexity (batch size 1, state carried across lines) ---------------------
def stream_log_probs(stream):
    """log P of every target of a split's stream (positions 1, 2, ...), read in order with batch size 1."""
    model.eval()
    source = stream.view(-1, 1).to(DEVICE)
    hidden, out = model.init_hidden(1), []
    with torch.no_grad():
        for i in range(0, source.size(0) - 1, BPTT):
            x, y = get_batch(source, i)
            logits, hidden = model(x, hidden)
            out.append(F.log_softmax(logits, dim=-1).gather(2, y.unsqueeze(2)).view(-1).cpu())
    return torch.cat(out).double().numpy()


awd_logprobs = {split: stream_log_probs(ids[split]) for split in ("valid", "test")}
for split, lp in awd_logprobs.items():
    assert len(lp) == sum(len(line) + 1 for line in data[split])     # the same targets as Task 1
    np.save(TASK2_DIR / f"{split}_token_logprobs.npy", lp)          # used by Task 3
awd_ppl = {split: math.exp(-lp.mean()) for split, lp in awd_logprobs.items()}
print(f"Step 6 — AWD-LSTM perplexity: validation {awd_ppl['valid']:.2f}, test {awd_ppl['test']:.2f} "
      f"({len(awd_logprobs['test']):,} test tokens, same as Task 1)")


# ---- Step 6: BLiMP (same paradigms, same length-normalized score and tie rule as Task 1) ---
def sentence_token_log_probs(sentences, batch_size=100):
    """Per-token log P (tokens + </s>) of each sentence on its own: zero initial state, </s> as the first
    input, same tokenization / <UNK> mapping as Task 1. Sentences are right-padded; the LSTM reads left to
    right, so padding never changes the probabilities of real tokens."""
    seqs = [[EOS_ID] + [stoi[tok] for tok in to_vocab(tokenize(s))] + [EOS_ID] for s in sentences]
    out = []
    model.eval()
    with torch.no_grad():
        for b in range(0, len(seqs), batch_size):
            chunk = seqs[b:b + batch_size]
            batch = torch.full((max(map(len, chunk)), len(chunk)), EOS_ID, dtype=torch.long)
            for j, s in enumerate(chunk):
                batch[:len(s), j] = torch.tensor(s)
            batch = batch.to(DEVICE)
            logits, _ = model(batch[:-1], model.init_hidden(len(chunk)))
            lp = F.log_softmax(logits, dim=-1).gather(2, batch[1:].unsqueeze(2)).squeeze(2).cpu().double().numpy()
            out.extend(lp[:len(s) - 1, j] for j, s in enumerate(chunk))
    return out


pair_rows = []
for name, pairs in blimp.items():
    good = sentence_token_log_probs([p["sentence_good"] for p in pairs])
    bad = sentence_token_log_probs([p["sentence_bad"] for p in pairs])
    for p, lp_good, lp_bad in zip(pairs, good, bad):
        diff = lp_good.mean() - lp_bad.mean()
        # a tie (no preference) if the two sentences are identical after <UNK> mapping or the scores are equal
        tie = (to_vocab(tokenize(p["sentence_good"])) == to_vocab(tokenize(p["sentence_bad"]))
               or abs(diff) <= BLIMP_TIE_TOL)
        pair_rows.append({"paradigm": name, "sentence_good": p["sentence_good"], "sentence_bad": p["sentence_bad"],
                          "score_good": lp_good.mean(), "score_bad": lp_bad.mean(),
                          "correct": diff > BLIMP_TIE_TOL and not tie, "tie": tie})
awd_pairs = pd.DataFrame(pair_rows)
awd_blimp = (awd_pairs.groupby("paradigm", sort=False)
             .agg(pairs=("correct", "size"), correct=("correct", "sum"), ties=("tie", "sum")).reset_index())
awd_blimp.insert(1, "category", awd_blimp["paradigm"].map(BLIMP_PARADIGMS))
awd_blimp["accuracy"] = awd_blimp["correct"] / awd_blimp["pairs"]
awd_blimp["KN accuracy (Task 1)"] = awd_blimp["paradigm"].map(dict(zip(blimp_table["paradigm"], blimp_table["accuracy"])))
awd_blimp_overall = awd_blimp["correct"].sum() / awd_blimp["pairs"].sum()

# ---- Results ----------------------------------------------------------------------
print("\nTable 2.1 — Training log (validation PPL with batch size 10, averaged weights once NT-ASGD has switched)")
print(history[["epoch", "train_ppl", "valid_ppl", "averaged", "minutes"]].to_string(
    index=False, float_format=lambda x: f"{x:,.2f}"))

print("\nTable 2.2 — Head-to-head on WikiText-2 (same vocabulary, same 244,980 test tokens)")
head_to_head = pd.DataFrame([
    {"model": f"Task 1: {best_variant}", "validation PPL": ppl[best_variant]["valid"],
     "test PPL": ppl[best_variant]["test"], "BLiMP overall": blimp_overall},
    {"model": f"Task 2: AWD-LSTM (best epoch {best_epoch})", "validation PPL": awd_ppl["valid"],
     "test PPL": awd_ppl["test"], "BLiMP overall": awd_blimp_overall},
])
print(head_to_head.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

print("\nTable 2.3 — BLiMP accuracy, AWD-LSTM (Task 1 Kneser–Ney alongside)")
print(awd_blimp.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"Overall BLiMP accuracy: {awd_blimp_overall:.3f} "
      f"({awd_blimp['correct'].sum():,} / {awd_blimp['pairs'].sum():,} pairs)")

print("\nTable 2.4 — Qualitative examples: the first correct and the first incorrect pair of every paradigm "
      "(score = mean log-probability per token)")
for name in BLIMP_PARADIGMS:
    rows = awd_pairs[awd_pairs["paradigm"] == name]
    for label, subset in (("correct", rows[rows["correct"]]), ("WRONG", rows[~rows["correct"]])):
        if len(subset):
            r = subset.iloc[0]
            print(f"  [{label:7s}] {name}\n      good {r['score_good']:7.3f}  {r['sentence_good']}\n"
                  f"      bad  {r['score_bad']:7.3f}  {r['sentence_bad']}")

awd_results = {
    "config": {"emb_size": EMB_SIZE, "hidden_size": HIDDEN_SIZE, "layers": N_LAYERS, "tied_weights": True,
               "dropconnect": DROPCONNECT, "embedding_dropout": EMB_DROPOUT, "output_dropout": OUT_DROPOUT,
               "optimizer": "NT-ASGD", "lr": LR, "nonmono": NONMONO, "batch_size": BATCH_SIZE, "bptt": BPTT,
               "clip": CLIP, "epochs": EPOCHS, "seed": SEED, "device": str(DEVICE)},
    "n_parameters": sum(p.numel() for p in params),
    "training_hours": history["minutes"].sum() / 60,
    "nt_asgd_switch_epoch": state["switch_epoch"],
    "best_epoch": best_epoch,
    "perplexity": awd_ppl,
    "blimp_per_paradigm": dict(zip(awd_blimp["paradigm"], awd_blimp["accuracy"])),
    "blimp_ties_per_paradigm": dict(zip(awd_blimp["paradigm"], awd_blimp["ties"].astype(int).tolist())),
    "blimp_overall": awd_blimp_overall,
}
(TASK2_DIR / "results.json").write_text(json.dumps(awd_results, indent=2))
history.to_csv(TASK2_DIR / "training_log.csv", index=False)
awd_blimp.to_csv(TASK2_DIR / "blimp.csv", index=False)
awd_pairs.to_csv(TASK2_DIR / "blimp_pairs.csv", index=False)
print(f"\nResults written to {TASK2_DIR}/, checkpoints in {CKPT_DIR}/  (cell time {(time.time() - t_start) / 60:.1f} min)")


Steps 1–2 — AWD-LSTM on cuda: layers 400->1150->1150->400, tied weights, 32.2M parameters
  resuming from checkpoints/task2_awd_lstm_h1150_last.pt (epochs done: 50)
Steps 3–5 — trained 50 epochs (2.8 h); NT-ASGD switch: after epoch 42; best validation loss at epoch 50
Step 6 — AWD-LSTM perplexity: validation 90.85, test 87.41 (244,980 test tokens, same as Task 1)

Table 2.1 — Training log (validation PPL with batch size 10, averaged weights once NT-ASGD has switched)
 epoch  train_ppl  valid_ppl  averaged  minutes
     1   1,494.21     661.16     False     7.63
     2     611.21     395.34     False     9.09
     3     425.93     306.83     False     8.99
     4     339.38     244.59     False     9.33
     5     276.46     218.59     False     8.60
     6     237.67     188.16     False     2.82
     7     208.25     178.01     False     2.80
     8     185.60     159.15     False     2.80
     9     170.50     152.32     False     2.79
    10     156.40     139.77     False     2.79


In [ ]:
# =====================================================================================
# Task 3 — Combine & Improve: Interpolated Hybrid Language Model (Kneser–Ney 4-gram + AWD-LSTM)
#   Step 1  classical half: Task 1's interpolated Kneser–Ney model (d = 0.75, λ tuned in Task 1)
#   Step 2  linear interpolation P_hybrid(w | context) = ω · P_KN(w | context) + (1 − ω) · P_AWD(w | context)
#   Step 3  ω tuned by grid search over {0.0, 0.1, ..., 1.0}, minimizing validation perplexity (never on test)
#   Step 4  test perplexity + BLiMP (same 10 paradigms, same length-normalized score and tie rule)
#   Step 5  final comparison table: Task 1 vs Task 2 vs Task 3, both metrics side by side
# Uses the models still in memory from Tasks 1 and 2 (run both cells first; once Task 2's training is done,
# its cell only loads the best checkpoint and evaluates). Both models give a distribution over the same
# V = 30,002 tokens for the same 244,980 test targets (every token plus one </s> per line), so they can be
# mixed token by token and the mixture is again a proper distribution. ω = 1 is exactly Task 1's model,
# ω = 0 exactly Task 2's.
# Seeds: Task 3 has no randomness (fixed grid, models from Tasks 1 and 2), so no new seed is needed.
# =====================================================================================
TASK3_DIR = Path("results/task3")
OMEGAS = [round(0.1 * i, 1) for i in range(11)]        # 0.0, 0.1, ..., 1.0
TASK3_DIR.mkdir(parents=True, exist_ok=True)
t_start = time.time()

# ---- Step 1: per-token log-probabilities of both halves ------------------------------------
kn_name = next(name for name, m in models.items() if isinstance(m, KneserNey))
kn_model = models[kn_name]


def kn_token_log_probs(lines):
    """KN log P of every target, in the same order as Task 2's stream (each token of a line, then </s>),
    and the longest n-gram ending in the target that occurs in training (4 = the exact 4-gram was seen,
    1 = only the word itself) — used only for the analysis below."""
    lps, longest = [], []
    for line in lines:
        for context, word in contexts_and_targets(line):
            lps.append(math.log(kn_model.prob(context, word)))
            longest.append(next((n for n in range(ORDER, 0, -1) if context[ORDER - n:] + (word,) in counts[n]), 0))
    return np.array(lps), np.array(longest)


kn_logprobs, longest_seen = {}, {}
for split in ("valid", "test"):
    kn_logprobs[split], longest_seen[split] = kn_token_log_probs(data[split])
    assert len(kn_logprobs[split]) == len(awd_logprobs[split])                  # the same targets
    assert math.isclose(math.exp(-kn_logprobs[split].mean()), ppl[kn_name][split], rel_tol=1e-9)
print(f"Step 1 — classical half: {kn_name}; neural half: AWD-LSTM (best epoch {best_epoch}); "
      f"{len(kn_logprobs['valid']):,} validation / {len(kn_logprobs['test']):,} test targets")


#Step 2: linear interpolation ------------------------------------------------------------
def hybrid_log_probs(lp_kn, lp_awd, omega):
    """log P_hybrid = log(ω · P_KN + (1 − ω) · P_AWD), computed with log-sum-exp so that small
    probabilities do not underflow."""
    with np.errstate(divide="ignore"):          # log 0 = -inf at ω = 0 or 1; logaddexp handles it exactly
        return np.logaddexp(np.log(omega) + lp_kn, np.log1p(-omega) + lp_awd)


def ppl_from(lp):
    return math.exp(-lp.mean())


# ---- Step 3: grid search for ω on validation --------------------------------------------
omega_grid = pd.DataFrame([
    {"ω": w,
     "validation PPL": ppl_from(hybrid_log_probs(kn_logprobs["valid"], awd_logprobs["valid"], w)),
     "test PPL": ppl_from(hybrid_log_probs(kn_logprobs["test"], awd_logprobs["test"], w))}
    for w in OMEGAS])
best_omega = float(omega_grid.loc[omega_grid["validation PPL"].idxmin(), "ω"])
# the end points of the grid are the two single models
assert math.isclose(omega_grid["test PPL"].iloc[0], awd_ppl["test"], rel_tol=1e-9)
assert math.isclose(omega_grid["test PPL"].iloc[-1], ppl[kn_name]["test"], rel_tol=1e-9)
hybrid_lp = {split: hybrid_log_probs(kn_logprobs[split], awd_logprobs[split], best_omega) for split in ("valid", "test")}
hybrid_ppl = {split: ppl_from(lp) for split, lp in hybrid_lp.items()}
for split, lp in hybrid_lp.items():
    np.save(TASK3_DIR / f"{split}_token_logprobs.npy", lp)
print(f"Step 3 — ω grid search on validation: best ω = {best_omega} (validation PPL {hybrid_ppl['valid']:.2f})")
print(f"Step 4 — hybrid perplexity: validation {hybrid_ppl['valid']:.2f}, test {hybrid_ppl['test']:.2f}")


# Step 4: BLiMP (same paradigms, same length-normalized score and tie rule as Tasks 1–2) -----
def blimp_token_log_probs(sentences):
    """Per-token log P (tokens + </s>) of each sentence on its own under both models: KN with <s> padding
    (as in Task 1), AWD-LSTM from a zero state with </s> as the first input (as in Task 2)."""
    kn_lp = [np.array(log_probs(kn_model, to_vocab(tokenize(s)))) for s in sentences]
    awd_lp = sentence_token_log_probs(sentences)
    assert all(len(a) == len(b) for a, b in zip(kn_lp, awd_lp))
    return kn_lp, awd_lp


blimp_items = [(name, pair) for name, pairs in blimp.items() for pair in pairs]
good_sentences = [pair["sentence_good"] for _, pair in blimp_items]
bad_sentences = [pair["sentence_bad"] for _, pair in blimp_items]
kn_good, awd_good = blimp_token_log_probs(good_sentences)
kn_bad, awd_bad = blimp_token_log_probs(bad_sentences)
identical = np.array([to_vocab(tokenize(g)) == to_vocab(tokenize(b)) for g, b in zip(good_sentences, bad_sentences)])


def blimp_decisions(omega):
    """Mean hybrid log-probability per token of every good and bad sentence, whether the pair is correct
    (good preferred by more than BLIMP_TIE_TOL) and whether it is a tie (no preference)."""
    score_good = np.array([hybrid_log_probs(k, a, omega).mean() for k, a in zip(kn_good, awd_good)])
    score_bad = np.array([hybrid_log_probs(k, a, omega).mean() for k, a in zip(kn_bad, awd_bad)])
    diff = score_good - score_bad
    tie = identical | (np.abs(diff) <= BLIMP_TIE_TOL)
    return score_good, score_bad, (diff > BLIMP_TIE_TOL) & ~tie, tie


# BLiMP accuracy over the whole ω grid — for analysis only; ω itself is chosen on validation perplexity
decisions = {w: blimp_decisions(w) for w in OMEGAS}
omega_grid["BLiMP overall"] = [decisions[w][2].mean() for w in OMEGAS]
print(f"  check: ω = 0 gives BLiMP {omega_grid['BLiMP overall'].iloc[0]:.3f} (Task 2: {awd_blimp_overall:.3f}), "
      f"ω = 1 gives {omega_grid['BLiMP overall'].iloc[-1]:.3f} (Task 1: {blimp_overall:.3f})")

score_good, score_bad, hybrid_correct, hybrid_tie = decisions[best_omega]
awd_score_good, awd_score_bad, awd_correct, _ = decisions[0.0]
kn_correct = decisions[1.0][2]
hybrid_pairs = pd.DataFrame({
    "paradigm": [name for name, _ in blimp_items], "sentence_good": good_sentences, "sentence_bad": bad_sentences,
    "score_good": score_good, "score_bad": score_bad, "correct": hybrid_correct, "tie": hybrid_tie,
    "awd_score_good": awd_score_good, "awd_score_bad": awd_score_bad,
    "awd_correct": awd_correct, "kn_correct": kn_correct})
hybrid_pairs["fixed_vs_awd"] = hybrid_pairs["correct"] & ~hybrid_pairs["awd_correct"]
hybrid_pairs["broken_vs_awd"] = ~hybrid_pairs["correct"] & hybrid_pairs["awd_correct"]
hybrid_blimp = (hybrid_pairs.groupby("paradigm", sort=False)
                .agg(pairs=("correct", "size"), **{"KN (Task 1)": ("kn_correct", "mean"),
                                                   "AWD (Task 2)": ("awd_correct", "mean"),
                                                   "hybrid (Task 3)": ("correct", "mean")},
                     ties=("tie", "sum"), fixed_vs_awd=("fixed_vs_awd", "sum"), broken_vs_awd=("broken_vs_awd", "sum"))
                .reset_index())
hybrid_blimp.insert(1, "category", hybrid_blimp["paradigm"].map(BLIMP_PARADIGMS))
hybrid_blimp_overall = hybrid_pairs["correct"].mean()

# Step 5: final comparison table ----------------------------------------------------------
comparison = pd.DataFrame([
    {"model": f"Task 1: {kn_name}", "validation PPL": ppl[kn_name]["valid"], "test PPL": ppl[kn_name]["test"],
     "BLiMP overall": blimp_overall},
    {"model": f"Task 2: AWD-LSTM (best epoch {best_epoch})", "validation PPL": awd_ppl["valid"],
     "test PPL": awd_ppl["test"], "BLiMP overall": awd_blimp_overall},
    {"model": f"Task 3: hybrid, ω = {best_omega}", "validation PPL": hybrid_ppl["valid"],
     "test PPL": hybrid_ppl["test"], "BLiMP overall": hybrid_blimp_overall},
])

# Analysis: where does the n-gram half help? ----------------------------------------------
# Test tokens grouped by the longest n-gram ending in the target that occurs in training. The best ω per
# group (on validation) is a diagnostic only: the group depends on the target word, so it cannot be used
# inside a model (the mixture would no longer sum to 1).
BUCKET_LABELS = {4: "exact 4-gram seen", 3: "longest seen: 3-gram", 2: "longest seen: 2-gram",
                 1: "only the word itself"}
bucket_rows = []
for n, label in BUCKET_LABELS.items():
    test_mask, valid_mask = longest_seen["test"] == n, longest_seen["valid"] == n
    bucket_rows.append({
        "test tokens": label, "share": test_mask.mean(),
        "KN PPL": ppl_from(kn_logprobs["test"][test_mask]), "AWD PPL": ppl_from(awd_logprobs["test"][test_mask]),
        "hybrid PPL": ppl_from(hybrid_lp["test"][test_mask]),
        "best ω for this group (valid)": min(OMEGAS, key=lambda w: ppl_from(hybrid_log_probs(
            kn_logprobs["valid"][valid_mask], awd_logprobs["valid"][valid_mask], w)))})
buckets = pd.DataFrame(bucket_rows)

# Qualitative examples: test tokens where mixing in KN changes log P the most compared with AWD alone
test_positions = [(i, j) for i, line in enumerate(data["test"]) for j in range(len(line) + 1)]
gain = hybrid_lp["test"] - awd_logprobs["test"]


def token_example(k, width=8):
    i, j = test_positions[k]
    line = data["test"][i]
    return {"context": ("… " if j > width else "") + " ".join(line[max(0, j - width):j]),
            "target": line[j] if j < len(line) else EOS,
            "P_KN": math.exp(kn_logprobs["test"][k]), "P_AWD": math.exp(awd_logprobs["test"][k]),
            "P_hybrid": math.exp(hybrid_lp["test"][k]), "Δ log P": gain[k]}


order = np.argsort(gain)
token_examples = pd.DataFrame([dict(token_example(k), kind="largest gain") for k in order[::-1][:10]]
                              + [dict(token_example(k), kind="largest loss") for k in order[:10]])

# ---- Results ---------------------------------------------------------------------------------
print("\nTable 3.1 — ω grid search (ω = weight of KN; ω chosen on validation PPL, test PPL and BLiMP for reference)")
print(omega_grid.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

print("\nTable 3.2 — Final comparison on WikiText-2 (same vocabulary, same 244,980 test tokens, same BLiMP pairs)")
print(comparison.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
print(f"  hybrid vs AWD-LSTM: test PPL {hybrid_ppl['test'] - awd_ppl['test']:+.2f} "
      f"({hybrid_ppl['test'] / awd_ppl['test'] - 1:+.1%}); hybrid vs KN: {hybrid_ppl['test'] - ppl[kn_name]['test']:+.2f}")

print("\nTable 3.3 — BLiMP accuracy per paradigm (fixed / broken = pairs the hybrid gets right / wrong that AWD alone got wrong / right)")
print(hybrid_blimp.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"Overall BLiMP accuracy, hybrid: {hybrid_blimp_overall:.3f} "
      f"({hybrid_pairs['correct'].sum():,} / {len(hybrid_pairs):,} pairs)")

print("\nTable 3.4 — Test perplexity by the longest n-gram (ending in the target) seen in training")
print(buckets.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
print(f"  tokens where the hybrid assigns higher probability than AWD alone: {(gain > 0).mean():.1%}")

print("\nTable 3.5 — Qualitative examples: test tokens with the largest gain / loss of log P from adding KN to AWD")
for kind, rows in token_examples.groupby("kind", sort=False):
    print(f"  {kind}:")
    for _, r in rows.iterrows():
        print(f"    P_KN {r['P_KN']:9.2e}  P_AWD {r['P_AWD']:9.2e}  P_hybrid {r['P_hybrid']:9.2e}  "
              f"{r['context']} [{r['target']}]")

print("\nTable 3.6 — Qualitative examples: BLiMP pairs whose decision changes when KN is mixed in "
      "(score = mean log-probability per token, AWD -> hybrid)")
for label, column in (("fixed", "fixed_vs_awd"), ("broken", "broken_vs_awd")):
    for _, r in hybrid_pairs[hybrid_pairs[column]].groupby("paradigm", sort=False).head(1).iterrows():
        print(f"  [{label:6s}] {r['paradigm']}\n"
              f"      good {r['awd_score_good']:7.3f} -> {r['score_good']:7.3f}  {r['sentence_good']}\n"
              f"      bad  {r['awd_score_bad']:7.3f} -> {r['score_bad']:7.3f}  {r['sentence_bad']}")

hybrid_results = {
    "classical_model": kn_name,
    "neural_model": f"AWD-LSTM (best epoch {best_epoch})",
    "omega_grid": omega_grid.to_dict(orient="records"),
    "best_omega": best_omega,
    "perplexity": hybrid_ppl,
    "blimp_per_paradigm": dict(zip(hybrid_blimp["paradigm"], hybrid_blimp["hybrid (Task 3)"])),
    "blimp_ties_per_paradigm": dict(zip(hybrid_blimp["paradigm"], hybrid_blimp["ties"].astype(int).tolist())),
    "blimp_overall": hybrid_blimp_overall,
    "comparison": comparison.to_dict(orient="records"),
}
(TASK3_DIR / "results.json").write_text(json.dumps(hybrid_results, indent=2, ensure_ascii=False, default=float))
omega_grid.to_csv(TASK3_DIR / "omega_grid.csv", index=False)
comparison.to_csv(TASK3_DIR / "comparison.csv", index=False)
hybrid_blimp.to_csv(TASK3_DIR / "blimp.csv", index=False)
hybrid_pairs.to_csv(TASK3_DIR / "blimp_pairs.csv", index=False)
buckets.to_csv(TASK3_DIR / "ngram_buckets.csv", index=False)
token_examples.to_csv(TASK3_DIR / "token_examples.csv", index=False)
print(f"\nResults written to {TASK3_DIR}/  (cell time {(time.time() - t_start) / 60:.1f} min)")

Step 1 — classical half: Kneser–Ney (d=0.75, λ=0.8); neural half: AWD-LSTM (best epoch 50); 217,431 validation / 244,980 test targets
Step 3 — ω grid search on validation: best ω = 0.1 (validation PPL 86.88)
Step 4 — hybrid perplexity: validation 86.88, test 84.22
  check: ω = 0 gives BLiMP 0.774 (Task 2: 0.774), ω = 1 gives 0.629 (Task 1: 0.629)

Table 3.1 — ω grid search (ω = weight of KN; ω chosen on validation PPL, test PPL and BLiMP for reference)
    ω  validation PPL  test PPL  BLiMP overall
0.000          90.848    87.412          0.774
0.100          86.879    84.216          0.752
0.200          87.819    85.259          0.743
0.300          90.318    87.756          0.734
0.400          94.133    91.502          0.726
0.500          99.410    96.645          0.716
0.600         106.587   103.609          0.707
0.700         116.574   113.264          0.701
0.800         131.370   127.504          0.690
0.900         156.642   151.672          0.679
1.000         243.954   23